In [7]:
import os
import joblib
import pandas as pd
import tensorflow as tf
from tensorflow import keras

# Ocultar advertencias de TensorFlow para mantener la terminal limpia
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 

MODEL_PATH = 'best_model_gpa_imp.keras'
SCALER_PATH = 'scaler_gpa.pkl'
QUANTILE_SCALER_PATH = 'quantile_scaler_gpa.pkl'

print("--- SISTEMA DE PREDICCIÓN DE GPA ---")
print("Cargando componentes...")

# 1. CARGA DE MODELO Y TRANSFORMADORES
try:
    model = keras.models.load_model(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    quantile_scaler = joblib.load(QUANTILE_SCALER_PATH)
    print("[OK] Modelo y escaladores cargados correctamente.")
except Exception as e:
    print(f"[ERROR] Problema al cargar los archivos: {e}")
    print("Asegúrate de tener best_model_gpa_imp.keras, scaler_gpa.pkl y quantile_scaler_gpa.pkl en esta carpeta.")
    exit()

print("-" * 36)

# 2. FUNCIONES AUXILIARES PARA INPUTS
def obtener_input_numerico(prompt, tipo=float):
    while True:
        try:
            valor = input(prompt)
            return tipo(valor)
        except ValueError:
            print(f"[!] Entrada inválida. Se requiere un valor tipo {tipo.__name__}.")

def obtener_input_categorico(prompt, opciones_validas):
    while True:
        valor = input(prompt).strip().lower()
        if valor in opciones_validas:
            return valor
        print(f"[!] Opción inválida. Elige una de las siguientes: {opciones_validas}")

# 3. LÓGICA DE CAPTURA Y PREDICCIÓN
def iniciar_prediccion():
    print("\nPor favor, ingresa los datos del estudiante:\n")
    
    # Inputs Numéricos Directos
    age = obtener_input_numerico("1. Edad del adolescente (ej. 16): ", int)
    social_media = obtener_input_numerico("2. Horas diarias en redes sociales (ej. 3.5): ", float)
    sleep = obtener_input_numerico("3. Horas de sueño (ej. 7.5): ", float)
    screen_sleep = obtener_input_numerico("4. Horas de pantalla antes de dormir (ej. 1.0): ", float)
    physical = obtener_input_numerico("5. Horas de actividad física (ej. 1.0): ", float)
    stress = obtener_input_numerico("6. Nivel de estrés (numérico, ej. 3): ", int)
    anxiety = obtener_input_numerico("7. Nivel de ansiedad (numérico, ej. 2): ", int)
    addiction = obtener_input_numerico("8. Nivel de adicción (numérico, ej. 1): ", int)

    # Inputs Categóricos (Requieren Feature Engineering)
    gender = obtener_input_categorico("9. Género (male / female): ", ['male', 'female'])
    platform = obtener_input_categorico("10. Plataforma principal (instagram / tiktok / both): ", ['instagram', 'tiktok', 'both'])
    social_int = obtener_input_categorico("11. Nivel de interacción social (low / medium / high): ", ['low', 'medium', 'high'])

    print("\nProcesando el Feature Engineering...")

    # --- FEATURE ENGINEERING MANUAL ---
    
    # A. Ordinal Mapping
    map_social = {'low': 0, 'medium': 1, 'high': 2}
    social_interaction_level = map_social[social_int]

    # B. One-Hot Encoding
    gender_male = 1 if gender == 'male' else 0
    platform_usage_Instagram = 1 if platform == 'instagram' else 0
    platform_usage_TikTok = 1 if platform == 'tiktok' else 0

    # Construcción del DataFrame en el ORDEN EXACTO del entrenamiento (X_test_ready.dtypes)
    columnas_ordenadas = [
        'age', 'daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep', 
        'physical_activity', 'social_interaction_level', 'stress_level', 
        'anxiety_level', 'addiction_level', 'gender_male', 
        'platform_usage_Instagram', 'platform_usage_TikTok'
    ]
    
    datos_crudos = [[
        age, social_media, sleep, screen_sleep, physical, 
        social_interaction_level, stress, anxiety, addiction, 
        gender_male, platform_usage_Instagram, platform_usage_TikTok
    ]]
    
    df_input = pd.DataFrame(datos_crudos, columns=columnas_ordenadas)

    try:
        # C. Aplicar Escalamientos a las columnas correctas
        col_standard = ['age', 'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level']
        col_quantile = ['daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep', 'physical_activity']
        
        # Transformamos utilizando los escaladores previamente entrenados
        df_input[col_standard] = scaler.transform(df_input[col_standard])
        df_input[col_quantile] = quantile_scaler.transform(df_input[col_quantile])
        
        # D. Predicción
        prediccion_raw = model.predict(df_input, verbose=0)
        gpa_predicho = prediccion_raw[0][0]
        
        # Acotar dentro del rango máximo de GPA (Ajustar si tu escala difiere de 0 a 4.0)
        gpa_final = max(0.0, min(4.0, gpa_predicho))
        
        print("=" * 40)
        print("        RESULTADO DE LA PREDICCIÓN")
        print("=" * 40)
        print(f"  > Rendimiento Académico (GPA): {gpa_final:.3f}")
        print("=" * 40 + "\n")
        
    except Exception as e:
        print(f"\n[ERROR] Ocurrió un problema en la transformación o predicción: {e}")

# Ejecutar el flujo
if __name__ == "__main__":
    iniciar_prediccion()

--- SISTEMA DE PREDICCIÓN DE GPA ---
Cargando componentes...
[OK] Modelo y escaladores cargados correctamente.
------------------------------------

Por favor, ingresa los datos del estudiante:


Procesando el Feature Engineering...
        RESULTADO DE LA PREDICCIÓN
  > Rendimiento Académico (GPA): 3.053

